# Modeling Exam (Medium) — CoderPad Practice Problems

Realistic CoderPad problems. 30 min each. All formulas provided — the challenge is clean execution under pressure, not memorization.

**Rules:**
- Set a 30-minute timer per problem
- No docs, no autocomplete, no outside help
- Run the test cell — all assertions must pass
- If you finish early, review edge cases

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
from typing import Any

---

## Problem 1: Multi-Head Self-Attention

Implement multi-head self-attention from scratch.

### Formula

```
Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
```

### Architecture

1. **Single combined projection:** One linear layer `W_qkv` projects input `x` from `d_model` → `3 * d_model` (no bias). Split the output into Q, K, V along the last dimension.
2. **Reshape to heads:** (B, T, d_model) → (B, num_heads, T, d_k) where `d_k = d_model // num_heads`
3. **Scaled dot-product attention** per head using the formula above
4. **Concatenate heads:** (B, num_heads, T, d_k) → (B, T, d_model)
5. **Output projection:** `W_out` linear layer `d_model → d_model` (no bias)

### Causal Mask

When `causal=True`, set attention scores where `query_pos < key_pos` to `-inf` before softmax. This prevents each position from attending to future positions.

### Constraints

- Assert `d_model % num_heads == 0` in `__init__`
- Do NOT use `nn.MultiheadAttention` or `F.scaled_dot_product_attention`
- No bias in any linear layers

### Interface

```python
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        # x: (B, T, d_model) -> output: (B, T, d_model)
```

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 1 ---
torch.manual_seed(42)

mha = MultiHeadSelfAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)

# Shape preservation
out = mha(x)
assert out.shape == (2, 10, 64), f"Expected (2,10,64), got {out.shape}"

# Causal forward
out_causal = mha(x, causal=True)
assert out_causal.shape == (2, 10, 64)

# Causal property: output at position t must not depend on positions > t
x_test = torch.randn(1, 5, 64)
out1 = mha(x_test, causal=True)
x_modified = x_test.clone()
x_modified[0, 3:, :] = torch.randn(2, 64)  # Change positions 3 and 4
out2 = mha(x_modified, causal=True)
assert torch.allclose(out1[0, :3], out2[0, :3], atol=1e-5), "Causal mask broken: early positions changed"
assert not torch.allclose(out1[0, 3], out2[0, 3], atol=1e-3), "Position 3 should differ"

# Parameter count: W_qkv (d * 3d, no bias) + W_out (d * d, no bias)
total_params = sum(p.numel() for p in mha.parameters())
assert total_params == 64 * 64 * 3 + 64 * 64, f"Unexpected param count: {total_params}"

# Gradient flow
x_grad = torch.randn(2, 10, 64, requires_grad=True)
out_grad = mha(x_grad)
out_grad.sum().backward()
assert x_grad.grad is not None, "Gradient should flow to input"
for name, p in mha.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"

# Invalid config
try:
    MultiHeadSelfAttention(d_model=63, num_heads=8)
    assert False, "Should have raised error for d_model not divisible by num_heads"
except (AssertionError, ValueError):
    pass

# Single head should work
mha_single = MultiHeadSelfAttention(d_model=32, num_heads=1)
assert mha_single(torch.randn(1, 5, 32)).shape == (1, 5, 32)

# Batch size 1
assert mha(torch.randn(1, 3, 64)).shape == (1, 3, 64)

print("Problem 1: ALL TESTS PASSED")

---

## Problem 2: Transformer Block

Build a complete pre-norm transformer block.

### Architecture

```
Pre-Norm Transformer Block:

  x ─────────────────────────────┐
  │                              │ (residual)
  LayerNorm                      │
  │                              │
  MultiHeadSelfAttention         │
  │                              │
  + ◄────────────────────────────┘
  │
  x' ────────────────────────────┐
  │                              │ (residual)
  LayerNorm                      │
  │                              │
  MLP (Linear → GELU → Linear)  │
  │                              │
  + ◄────────────────────────────┘
  │
  output
```

### MLP Details

- `Linear(d_model, 4 * d_model)` → `GELU` → `Linear(4 * d_model, d_model)`

### Requirements

- Use `nn.LayerNorm` for normalization
- Use your `MultiHeadSelfAttention` from Problem 1 (or `nn.MultiheadAttention` if you haven't solved Problem 1)
- Support optional `causal` flag passed through to attention

### Interface

```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        # x: (B, T, d_model) -> output: (B, T, d_model)
```

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor, causal: bool = False) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 2 ---
torch.manual_seed(42)

block = TransformerBlock(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)

# Shape preservation
out = block(x)
assert out.shape == (2, 10, 64), f"Expected (2,10,64), got {out.shape}"

# Causal mode
out_causal = block(x, causal=True)
assert out_causal.shape == (2, 10, 64)

# Residual connection: output should not be identical to input (attention changes it)
# but should be close-ish due to residual
assert not torch.allclose(out, x, atol=1e-3), "Output should differ from input"

# Residual connection check: if we zero out all attention/MLP weights,
# output should equal input (residual passthrough)
block_zero = TransformerBlock(d_model=64, num_heads=8)
with torch.no_grad():
    for name, p in block_zero.named_parameters():
        if 'weight' in name and 'norm' not in name and 'ln' not in name:
            p.zero_()
        if 'bias' in name and 'norm' not in name and 'ln' not in name:
            p.zero_()
x_res = torch.randn(1, 5, 64)
out_res = block_zero(x_res)
assert torch.allclose(out_res, x_res, atol=1e-5), "With zeroed weights, residual should pass through"

# Gradient flow through all parameters
x_grad = torch.randn(2, 10, 64, requires_grad=True)
out_grad = block(x_grad)
out_grad.sum().backward()
assert x_grad.grad is not None, "Gradient should flow to input"
for name, p in block.named_parameters():
    assert p.grad is not None, f"No gradient for {name}"

# Check that MLP has 4x expansion
# Count params: 2 LayerNorms (64*2 each) + attention (64*64*4) + MLP (64*256 + 256 + 256*64 + 64)
param_names = [name for name, _ in block.named_parameters()]
has_mlp = any('mlp' in name.lower() or 'ff' in name.lower() or 'linear' in name.lower() for name in param_names)
total_params = sum(p.numel() for p in block.parameters())
assert total_params > 64 * 64 * 4, "Should have attention + MLP + LayerNorm params"

# Different sequence lengths should work
assert block(torch.randn(1, 1, 64)).shape == (1, 1, 64)
assert block(torch.randn(1, 100, 64)).shape == (1, 100, 64)

print("Problem 2: ALL TESTS PASSED")

---

## Problem 3: DDPM Noise Schedule + Forward Process

Implement the core diffusion components from DDPM (Denoising Diffusion Probabilistic Models).

### Formulas

**Linear beta schedule:**
```
beta_t = beta_start + (beta_end - beta_start) * t / (T - 1)    for t = 0, 1, ..., T-1
```

**Derived quantities:**
```
alpha_t = 1 - beta_t
alpha_bar_t = product(alpha_0, alpha_1, ..., alpha_t)   (cumulative product)
```

**Forward process (adding noise):**
```
q(x_t | x_0) = N(x_t; sqrt(alpha_bar_t) * x_0, (1 - alpha_bar_t) * I)

Sampling: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * noise
    where noise ~ N(0, I)
```

**Training loss (simplified):**
```
L = MSE(noise_predicted, noise_actual)
```

### Requirements

**`DDPMSchedule`:**
- `__init__(self, num_timesteps: int, beta_start: float = 1e-4, beta_end: float = 0.02)`
  - Compute and store `betas`, `alphas`, `alpha_bars` as 1D tensors of length `num_timesteps`
- `q_sample(self, x_0: Tensor, t: Tensor, noise: Tensor | None = None) -> tuple[Tensor, Tensor]`
  - `x_0`: (B, ...) clean data
  - `t`: (B,) integer timesteps
  - `noise`: optional, same shape as x_0. If None, sample from N(0, I)
  - Return `(x_t, noise)` — the noised sample and the noise used
- `compute_loss(self, model_output: Tensor, noise: Tensor) -> Tensor`
  - MSE between predicted noise and actual noise (scalar)

In [ ]:
class DDPMSchedule:
    def __init__(self, num_timesteps: int, beta_start: float = 1e-4, beta_end: float = 0.02):
        # YOUR CODE HERE
        pass

    def q_sample(
        self, x_0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor | None = None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # YOUR CODE HERE
        pass

    def compute_loss(self, model_output: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 3 ---
torch.manual_seed(42)

schedule = DDPMSchedule(num_timesteps=1000)

# Beta schedule shape and range
assert schedule.betas.shape == (1000,), f"Expected (1000,), got {schedule.betas.shape}"
assert abs(schedule.betas[0].item() - 1e-4) < 1e-6, "First beta should be beta_start"
assert abs(schedule.betas[-1].item() - 0.02) < 1e-6, "Last beta should be beta_end"
assert (schedule.betas[1:] >= schedule.betas[:-1]).all(), "Betas should be monotonically increasing"

# Alpha and alpha_bar
assert schedule.alphas.shape == (1000,)
assert schedule.alpha_bars.shape == (1000,)
assert torch.allclose(schedule.alphas, 1.0 - schedule.betas)
assert schedule.alpha_bars[0] > schedule.alpha_bars[-1], "alpha_bar should decrease over time"
assert abs(schedule.alpha_bars[0].item() - schedule.alphas[0].item()) < 1e-6, "alpha_bar_0 = alpha_0"

# Forward process: at t=0, output should be very close to input
x_0 = torch.randn(4, 3, 8, 8)  # batch of small "images"
t_zero = torch.zeros(4, dtype=torch.long)
x_t, noise = schedule.q_sample(x_0, t_zero)
assert x_t.shape == x_0.shape, "q_sample should preserve shape"
assert torch.allclose(x_t, x_0, atol=0.05), "At t=0, x_t should be ~x_0"

# Forward process: at t=999, output should be ~noise
t_max = torch.full((4,), 999, dtype=torch.long)
x_t_noisy, noise_max = schedule.q_sample(x_0, t_max)
# At t=999, alpha_bar is very small, so x_t should be mostly noise
signal_ratio = schedule.alpha_bars[999].sqrt().item()
assert signal_ratio < 0.1, f"At t=999, signal should be very small, got {signal_ratio}"

# Deterministic with provided noise
fixed_noise = torch.randn_like(x_0)
x_t1, n1 = schedule.q_sample(x_0, torch.tensor([500, 500, 500, 500]), noise=fixed_noise)
x_t2, n2 = schedule.q_sample(x_0, torch.tensor([500, 500, 500, 500]), noise=fixed_noise)
assert torch.allclose(x_t1, x_t2), "Same noise should give same result"
assert torch.allclose(n1, fixed_noise), "Should return the noise used"

# Loss computation
pred = torch.randn(4, 3, 8, 8)
target = torch.randn(4, 3, 8, 8)
loss = schedule.compute_loss(pred, target)
assert loss.ndim == 0, "Loss should be scalar"
expected_loss = F.mse_loss(pred, target)
assert torch.allclose(loss, expected_loss), "Loss should be MSE"

print("Problem 3: ALL TESTS PASSED")

---

## Problem 4: Simple Training Loop

Implement a training loop with early stopping.

### Requirements

**`Trainer`:**
- `__init__(self, model, optimizer, train_loader, val_loader)`
- `train_epoch(self) -> float` — run one training epoch, return mean loss
  - For each batch: `forward → loss → backward → optimizer.step() → optimizer.zero_grad()`
  - The model's `forward` takes a batch (tuple of tensors) and returns `(loss, logits)`
  - Set model to train mode
- `evaluate(self) -> float` — run validation, return mean loss
  - No gradient computation (`torch.no_grad()`)
  - Set model to eval mode
- `fit(self, num_epochs: int, patience: int) -> dict` — train with early stopping
  - After each epoch, evaluate on validation set
  - If val loss hasn't improved for `patience` consecutive epochs, stop early
  - Track the best val loss and which epoch it occurred at (0-indexed)
  - Return:
    ```python
    {
        "train_losses": list[float],
        "val_losses": list[float],
        "best_epoch": int,
        "stopped_early": bool
    }
    ```

In [ ]:
class Trainer:
    def __init__(self, model: nn.Module, optimizer: torch.optim.Optimizer,
                 train_loader: DataLoader, val_loader: DataLoader):
        # YOUR CODE HERE
        pass

    def train_epoch(self) -> float:
        # YOUR CODE HERE
        pass

    def evaluate(self) -> float:
        # YOUR CODE HERE
        pass

    def fit(self, num_epochs: int, patience: int) -> dict:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 4 ---

# Simple regression model for testing
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 1)
    def forward(self, batch):
        x, y = batch
        pred = self.linear(x).squeeze(-1)
        loss = F.mse_loss(pred, y)
        return loss, pred

torch.manual_seed(42)
model = SimpleModel()

# Create data with learnable pattern
X = torch.randn(200, 10)
true_w = torch.randn(10)
y = X @ true_w + 0.1 * torch.randn(200)

train_ds = torch.utils.data.TensorDataset(X[:160], y[:160])
val_ds = torch.utils.data.TensorDataset(X[160:], y[160:])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
trainer = Trainer(model, optimizer, train_loader, val_loader)

# Single epoch
train_loss = trainer.train_epoch()
assert isinstance(train_loss, float), "train_epoch should return float"
assert train_loss > 0, "Loss should be positive"

val_loss = trainer.evaluate()
assert isinstance(val_loss, float), "evaluate should return float"
assert val_loss > 0, "Val loss should be positive"

# Full training with early stopping
torch.manual_seed(42)
model2 = SimpleModel()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.01)
trainer2 = Trainer(model2, optimizer2, train_loader, val_loader)

result = trainer2.fit(num_epochs=50, patience=5)
assert "train_losses" in result
assert "val_losses" in result
assert "best_epoch" in result
assert "stopped_early" in result
assert len(result["train_losses"]) == len(result["val_losses"])

# Loss should decrease
assert result["train_losses"][-1] < result["train_losses"][0], "Training loss should decrease"

# If stopped early, should have run fewer than 50 epochs
if result["stopped_early"]:
    assert len(result["train_losses"]) < 50
    assert len(result["train_losses"]) <= result["best_epoch"] + 5 + 1

# best_epoch should be valid
assert 0 <= result["best_epoch"] < len(result["val_losses"])
best_val = result["val_losses"][result["best_epoch"]]
assert best_val == min(result["val_losses"]), "best_epoch should have the lowest val loss"

print("Problem 4: ALL TESTS PASSED")

---

## Problem 5: Convolutional Feature Extractor

Build a small CNN feature extractor.

### Architecture

3 convolutional blocks, each consisting of:
```
Conv2d(in_ch, out_ch, kernel_size=3, padding=1) → BatchNorm2d(out_ch) → ReLU → MaxPool2d(2)
```

Followed by:
```
AdaptiveAvgPool2d((1, 1)) → Flatten
```

### Channel Progression

Default: `[3, 32, 64, 128]` — meaning:
- Block 1: 3 → 32
- Block 2: 32 → 64
- Block 3: 64 → 128

The final output dimension is the last channel count (128 by default).

### Interface

```python
class ConvFeatureExtractor(nn.Module):
    def __init__(self, channels: list[int] | None = None):
        # channels defaults to [3, 32, 64, 128]
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 3, H, W) -> output: (B, final_channels)
    @property
    def output_dim(self) -> int:
        # Return final channel count
```

In [ ]:
class ConvFeatureExtractor(nn.Module):
    def __init__(self, channels: list[int] | None = None):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # YOUR CODE HERE
        pass

    @property
    def output_dim(self) -> int:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 5 ---
torch.manual_seed(42)

cnn = ConvFeatureExtractor()

# Default channels
assert cnn.output_dim == 128

# Standard input
x = torch.randn(4, 3, 32, 32)
out = cnn(x)
assert out.shape == (4, 128), f"Expected (4, 128), got {out.shape}"

# Different spatial sizes (adaptive pooling should handle it)
x_big = torch.randn(2, 3, 224, 224)
out_big = cnn(x_big)
assert out_big.shape == (2, 128), f"Expected (2, 128), got {out_big.shape}"

x_small = torch.randn(1, 3, 8, 8)
out_small = cnn(x_small)
assert out_small.shape == (1, 128), f"Expected (1, 128), got {out_small.shape}"

# Custom channels
cnn_custom = ConvFeatureExtractor(channels=[3, 16, 32])
assert cnn_custom.output_dim == 32
out_custom = cnn_custom(torch.randn(2, 3, 32, 32))
assert out_custom.shape == (2, 32)

# Gradient flow
x_grad = torch.randn(2, 3, 32, 32, requires_grad=True)
out_grad = cnn(x_grad)
out_grad.sum().backward()
assert x_grad.grad is not None, "Gradient should flow to input"

# Output should be 2D (batch, features) not 4D
assert out.ndim == 2, "Output should be flattened to 2D"

# BatchNorm behavior: eval mode should give deterministic output
cnn.eval()
x_test = torch.randn(2, 3, 32, 32)
out1 = cnn(x_test)
out2 = cnn(x_test)
assert torch.allclose(out1, out2), "Eval mode should be deterministic"
cnn.train()

print("Problem 5: ALL TESTS PASSED")

---

## Problem 6: Weight Initialization

Implement common weight initialization schemes and a utility to apply them to a model.

### Formulas

**Xavier Uniform:**
```
bound = sqrt(6 / (fan_in + fan_out))
W ~ Uniform(-bound, bound)
```

**Kaiming Normal:**
```
std = sqrt(2 / fan_in)
W ~ Normal(0, std)
```

**Zero Init:**
```
W = 0
```

Where:
- `fan_in` = number of input features (for Linear: `in_features`, for Conv2d: `in_channels * kernel_h * kernel_w`)
- `fan_out` = number of output features (for Linear: `out_features`, for Conv2d: `out_channels * kernel_h * kernel_w`)

### Requirements

**Initialization functions** (operate in-place on a weight tensor):
- `xavier_uniform_init(tensor: Tensor, fan_in: int, fan_out: int)` — no `nn.init`
- `kaiming_normal_init(tensor: Tensor, fan_in: int)` — no `nn.init`
- `zero_init(tensor: Tensor)` — no `nn.init`

**`apply_init(model: nn.Module, method: str)`:**
- Walk all `nn.Linear` and `nn.Conv2d` layers in the model
- Apply the specified method (`"xavier"`, `"kaiming"`, or `"zero"`) to the `.weight` tensor
- If a layer has a `.bias`, always zero-initialize it
- Compute `fan_in` and `fan_out` automatically from the layer type

In [ ]:
def xavier_uniform_init(tensor: torch.Tensor, fan_in: int, fan_out: int) -> None:
    """In-place Xavier uniform init. No nn.init."""
    # YOUR CODE HERE
    pass


def kaiming_normal_init(tensor: torch.Tensor, fan_in: int) -> None:
    """In-place Kaiming normal init. No nn.init."""
    # YOUR CODE HERE
    pass


def zero_init(tensor: torch.Tensor) -> None:
    """In-place zero init. No nn.init."""
    # YOUR CODE HERE
    pass


def apply_init(model: nn.Module, method: str) -> None:
    """Apply init to all Linear and Conv2d layers. method: 'xavier', 'kaiming', or 'zero'."""
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 6 ---
torch.manual_seed(42)

# Test Xavier uniform
w = torch.empty(100, 200)
xavier_uniform_init(w, fan_in=200, fan_out=100)
bound = math.sqrt(6.0 / (200 + 100))
assert w.min().item() >= -bound - 0.01, f"Xavier min {w.min().item()} out of range"
assert w.max().item() <= bound + 0.01, f"Xavier max {w.max().item()} out of range"
assert abs(w.mean().item()) < 0.05, "Xavier should be roughly zero-mean"

# Test Kaiming normal
w2 = torch.empty(100, 200)
kaiming_normal_init(w2, fan_in=200)
expected_std = math.sqrt(2.0 / 200)
assert abs(w2.std().item() - expected_std) < 0.02, f"Kaiming std {w2.std().item()} vs expected {expected_std}"
assert abs(w2.mean().item()) < 0.05, "Kaiming should be roughly zero-mean"

# Test zero init
w3 = torch.randn(50, 50)
zero_init(w3)
assert (w3 == 0).all(), "Zero init should zero everything"

# Test apply_init with a model
class TestModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(64, 128)
        self.linear2 = nn.Linear(128, 10)
        self.conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.norm = nn.LayerNorm(128)  # Should be skipped

test_model = TestModel()

# Xavier init
apply_init(test_model, "xavier")
# Check linear layer
bound_l1 = math.sqrt(6.0 / (64 + 128))
assert test_model.linear1.weight.min().item() >= -bound_l1 - 0.01
assert test_model.linear1.weight.max().item() <= bound_l1 + 0.01
# Bias should be zeroed
assert (test_model.linear1.bias == 0).all(), "Bias should be zeroed"
# Conv layer should also be initialized
conv_fan_in = 3 * 3 * 3  # in_channels * kH * kW
conv_fan_out = 16 * 3 * 3
conv_bound = math.sqrt(6.0 / (conv_fan_in + conv_fan_out))
assert test_model.conv.weight.min().item() >= -conv_bound - 0.02
assert test_model.conv.weight.max().item() <= conv_bound + 0.02

# Kaiming init
apply_init(test_model, "kaiming")
expected_std_l1 = math.sqrt(2.0 / 64)
assert abs(test_model.linear1.weight.std().item() - expected_std_l1) < 0.05

# Zero init
apply_init(test_model, "zero")
assert (test_model.linear1.weight == 0).all()
assert (test_model.conv.weight == 0).all()

print("Problem 6: ALL TESTS PASSED")

---

## Problem 7: Gradient Accumulation Training Step

Implement gradient accumulation for training with large effective batch sizes on limited memory.

### Concept

Instead of one large batch, split it into `accum_steps` micro-batches:
1. For each micro-batch: compute loss, scale by `1 / accum_steps`, call backward
2. After all micro-batches: optionally clip gradients, then optimizer step, then zero grad

This accumulates gradients across micro-batches, giving the same result as a single large batch.

### Gradient Clipping

```
total_norm = sqrt(sum(p.grad.norm()^2 for all params))
if total_norm > max_norm:
    scale = max_norm / total_norm
    for each param: p.grad *= scale
```

### Requirements

**`gradient_accumulation_step(model, optimizer, micro_batches, loss_fn, max_norm=None) -> dict`:**
- `model`: nn.Module
- `optimizer`: torch.optim.Optimizer
- `micro_batches`: list of input tensors (each is one micro-batch)
- `loss_fn`: callable, takes `(model_output, input)` and returns a scalar loss
  - For simplicity, `loss_fn(model(x), x)` — e.g., autoencoder reconstruction
- `max_norm`: if not None, clip gradients before stepping
- Returns `{"total_loss": float, "grad_norm": float, "num_micro_batches": int}`
  - `total_loss`: average loss across all micro-batches (unscaled)
  - `grad_norm`: gradient norm BEFORE clipping (or just the norm if no clipping)

**`clip_grad_norm(model, max_norm) -> float`:**
- Implement gradient clipping as described above. No `torch.nn.utils.clip_grad_norm_`.
- Return the total gradient norm BEFORE clipping.

In [ ]:
def clip_grad_norm(model: nn.Module, max_norm: float) -> float:
    """Clip gradients by global norm. Return norm before clipping. No torch.nn.utils."""
    # YOUR CODE HERE
    pass


def gradient_accumulation_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    micro_batches: list[torch.Tensor],
    loss_fn: Any,
    max_norm: float | None = None,
) -> dict:
    # YOUR CODE HERE
    pass

In [ ]:
# --- Tests for Problem 7 ---
torch.manual_seed(42)

# Simple autoencoder for testing
class SimpleAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Linear(10, 5)
        self.dec = nn.Linear(5, 10)
    def forward(self, x):
        return self.dec(torch.relu(self.enc(x)))

model = SimpleAutoencoder()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = lambda output, target: F.mse_loss(output, target)

# Test gradient accumulation
micro_batches = [torch.randn(8, 10) for _ in range(4)]  # 4 micro-batches of 8
result = gradient_accumulation_step(model, optimizer, micro_batches, loss_fn)

assert "total_loss" in result
assert "grad_norm" in result
assert "num_micro_batches" in result
assert result["num_micro_batches"] == 4
assert result["total_loss"] > 0
assert result["grad_norm"] > 0

# Verify gradient accumulation matches single large batch
torch.manual_seed(42)
model_accum = SimpleAutoencoder()
model_single = SimpleAutoencoder()
# Copy weights
model_single.load_state_dict(model_accum.state_dict())

opt_accum = torch.optim.SGD(model_accum.parameters(), lr=0.01)
opt_single = torch.optim.SGD(model_single.parameters(), lr=0.01)

data = torch.randn(32, 10)
micro_batches_test = [data[i*8:(i+1)*8] for i in range(4)]

# Accumulation path
gradient_accumulation_step(model_accum, opt_accum, micro_batches_test, loss_fn)

# Single batch path
out_single = model_single(data)
loss_single = loss_fn(out_single, data)
loss_single.backward()
opt_single.step()
opt_single.zero_grad()

# Weights should be approximately equal
for (n1, p1), (n2, p2) in zip(model_accum.named_parameters(), model_single.named_parameters()):
    assert torch.allclose(p1, p2, atol=1e-5), f"Param {n1} differs between accum and single batch"

# Test gradient clipping
torch.manual_seed(42)
model_clip = SimpleAutoencoder()
opt_clip = torch.optim.SGD(model_clip.parameters(), lr=0.01)
result_clip = gradient_accumulation_step(
    model_clip, opt_clip, [torch.randn(8, 10) * 10], loss_fn, max_norm=1.0
)
assert result_clip["grad_norm"] > 0

# Test clip_grad_norm directly
model_cn = SimpleAutoencoder()
x_cn = torch.randn(8, 10) * 100  # large values for large gradients
loss_cn = loss_fn(model_cn(x_cn), x_cn)
loss_cn.backward()
norm_before = clip_grad_norm(model_cn, max_norm=1.0)
assert norm_before > 1.0, "Gradients should be large before clipping"
# Check norm after clipping
norm_after = sum(p.grad.norm().item() ** 2 for p in model_cn.parameters()) ** 0.5
assert abs(norm_after - 1.0) < 1e-4, f"After clipping, norm should be 1.0, got {norm_after}"

print("Problem 7: ALL TESTS PASSED")

---

## Problem 8: Cosine Similarity Loss (Contrastive / CLIP-style)

Implement the InfoNCE / CLIP-style contrastive loss for image-text pairs.

### Formula

Given a batch of `B` paired (image, text) embeddings:

```
# Normalize embeddings
img_emb = img_emb / ||img_emb||    (L2 normalize along feature dim)
text_emb = text_emb / ||text_emb||  (L2 normalize along feature dim)

# Compute logits
logits = (img_emb @ text_emb.T) * exp(log_temperature)

# Targets: diagonal is the matching pair
targets = [0, 1, 2, ..., B-1]

# Symmetric loss
loss = 0.5 * (CrossEntropy(logits, targets) + CrossEntropy(logits.T, targets))
```

The `log_temperature` is a **learnable scalar parameter** (initialized to `log(1/0.07)` as in CLIP).

### Requirements

**`CLIPLoss(nn.Module)`:**
- `__init__(self, init_temperature: float = 0.07)`
  - Store `log_temperature` as `nn.Parameter` initialized to `log(1 / init_temperature)`
- `forward(self, img_emb: Tensor, text_emb: Tensor) -> dict`
  - `img_emb`: (B, D) image embeddings
  - `text_emb`: (B, D) text embeddings
  - Return `{"loss": scalar, "logits": (B,B) tensor, "temperature": float}`
  - L2-normalize embeddings before computing logits
  - Use `F.cross_entropy` for the CE computation

In [ ]:
class CLIPLoss(nn.Module):
    def __init__(self, init_temperature: float = 0.07):
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(self, img_emb: torch.Tensor, text_emb: torch.Tensor) -> dict:
        # YOUR CODE HERE
        pass

In [ ]:
# --- Tests for Problem 8 ---
torch.manual_seed(42)

clip_loss = CLIPLoss(init_temperature=0.07)

# Check initial temperature
expected_log_temp = math.log(1.0 / 0.07)
assert abs(clip_loss.log_temperature.item() - expected_log_temp) < 1e-5, \
    f"Expected log_temp={expected_log_temp}, got {clip_loss.log_temperature.item()}"

# Basic forward
B, D = 8, 64
img_emb = torch.randn(B, D)
text_emb = torch.randn(B, D)
result = clip_loss(img_emb, text_emb)
assert "loss" in result
assert "logits" in result
assert "temperature" in result
assert result["logits"].shape == (B, B), f"Expected ({B},{B}), got {result['logits'].shape}"
assert result["loss"].ndim == 0, "Loss should be scalar"
assert result["loss"].item() > 0, "Loss should be positive"

# Temperature should be exp(log_temperature)
assert abs(result["temperature"] - math.exp(expected_log_temp)) < 1e-3

# Perfect matching: identical embeddings should give low loss
shared_emb = F.normalize(torch.randn(4, 32), dim=-1)
result_perfect = clip_loss(shared_emb, shared_emb)
# With identical normalized embeddings, diagonal of logits should be highest
logits_p = result_perfect["logits"]
assert (logits_p.argmax(dim=-1) == torch.arange(4)).all(), \
    "Diagonal should be highest for identical embeddings"

# Gradient flow to embeddings
img_grad = torch.randn(4, 32, requires_grad=True)
text_grad = torch.randn(4, 32, requires_grad=True)
result_grad = clip_loss(img_grad, text_grad)
result_grad["loss"].backward()
assert img_grad.grad is not None, "Gradient should flow to image embeddings"
assert text_grad.grad is not None, "Gradient should flow to text embeddings"

# Temperature is learnable
assert clip_loss.log_temperature.grad is not None, "Temperature should receive gradients"

# Logits should be symmetric in structure (not values, but loss uses both directions)
# The loss should be symmetric: swapping img and text should give same loss
torch.manual_seed(42)
clip_loss2 = CLIPLoss(init_temperature=0.07)
img_s = torch.randn(4, 16)
text_s = torch.randn(4, 16)
r1 = clip_loss2(img_s, text_s)
r2 = clip_loss2(text_s, img_s)
assert abs(r1["loss"].item() - r2["loss"].item()) < 1e-5, "Loss should be symmetric"

print("Problem 8: ALL TESTS PASSED")

---

## Scoring Yourself

| Result | Assessment |
|--------|------------|
| Solved in < 25 min, all tests pass | **Strong pass** — you'd ace this question |
| Solved in 25-30 min, minor debugging | **Pass** — exam-ready |
| 30-40 min or needed hints | **Borderline** — redo tomorrow |
| Couldn't solve or > 40 min | **Drill this** — break it into pieces, understand each part |

### Daily Practice

1. **Pick 1 problem (30 min):** Set timer, no docs, no autocomplete
2. **Run tests:** All assertions must pass
3. **Review (10 min):** If you failed, identify the sticking point (tensor shapes? API? math?) and note it for tomorrow

### Problem Difficulty Ranking

| Tier | Problems |
|------|----------|
| Start here | P4 (Training Loop), P5 (CNN), P6 (Weight Init) |
| Core | P1 (Attention), P2 (Transformer), P3 (DDPM) |
| Advanced | P7 (Grad Accumulation), P8 (CLIP Loss) |